# 11. 재정렬 개선 (v2.1): 정답 문단을 위로 올리기

09에서 쿼리 재작성·HyDE는 효과가 없었고, 실패 유형을 보니 **정답 문단이 후보 100개 안에는 있는데 재정렬이 위로 못 올린 경우**가 더 많았습니다 (겹침 하 0.236 / 중 0.187 / 상 0.108).
그래서 **같은 후보 100개**(bge dense)를 두고 재정렬만 바꿔 비교합니다.

| 재정렬 모델 | 크기 | 특징 |
|---|---|---|
| ms-marco-MiniLM-L-6-v2 (기준) | 22M | 지금 챗봇에서 쓰는 모델 |
| ms-marco-MiniLM-L-12-v2 | 33M | 같은 계열, 층 2배 |
| bge-reranker-base | 278M | 03에서 비교했던 모델 |
| bge-reranker-large | 560M | 더 큰 모델 |
| bge-reranker-v2-m3 | 568M | 최신 다국어 모델 |
| MedCPT-Cross-Encoder | 110M | **의료 특화** (PubMed 검색 기록으로 학습) |

합치기 (검증 200으로 설정 선택):
- **재정렬 + dense**: 질문별로 점수를 표준화한 뒤 `재정렬 + α × dense`
- **재정렬 2개 앙상블**: 두 모델의 표준화 점수 평균

보고는 시험 500 (답 있는 건강). 추가로 **답 없음 100 + 분야 밖 50**의 최고 점수로 거절 구분력(AUC)도 잽니다.
점수는 드라이브 `health/rerank2/scores_*.npy`에 저장돼서, 다시 돌리면 계산을 건너뜁니다. A100 기준 약 20분.

## 1. 설치 + 경로

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR   = '/content/drive/MyDrive/rag_msmarco/health'
OUT_DIR    = f'{SAVE_DIR}/rerank2'
REPORT_DIR = '/content/drive/MyDrive/RAG/2_results/retrieval_rerank2_v2.1'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

## 2. 설정

In [ ]:
import re, json, time, string, itertools
from collections import Counter
import numpy as np
import pandas as pd
import torch

K          = 100
EMB_MODEL  = 'BAAI/bge-base-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
BASE       = 'MiniLM-L-6 (기준)'
RERANKERS  = {   # 이름: (모델, max_length, 배치)
    BASE:              ('cross-encoder/ms-marco-MiniLM-L-6-v2', 256, 512),
    'MiniLM-L-12':     ('cross-encoder/ms-marco-MiniLM-L-12-v2', 256, 512),
    'bge-reranker-base':  ('BAAI/bge-reranker-base', 512, 256),
    'bge-reranker-large': ('BAAI/bge-reranker-large', 512, 128),
    'bge-reranker-v2-m3': ('BAAI/bge-reranker-v2-m3', 512, 128),
    'MedCPT (의료)':   ('ncbi/MedCPT-Cross-Encoder', 512, 256),
}
ALPHAS = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다 (A100 권장).'
print('GPU:', torch.cuda.get_device_name(0))

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

## 3. DB + dense 후보 100개 (모든 재정렬 모델이 같은 후보를 씀)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

t = time.time()
corpus   = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
val_set  = [q for q in load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl') if q.get('gold_pids')]
index    = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
texts    = [c['text'] for c in corpus]
assert index.ntotal == len(corpus)

test_set = [q for q in eval_set if q['domain'] == 'health' and q.get('answerable', True) and q.get('gold_pids')]
noans    = [q for q in eval_set if q['domain'] == 'health' and not q.get('answerable', True)]
other    = [q for q in eval_set if q['domain'] != 'health']
sets = {'val': val_set, 'test': test_set, 'noans': noans, 'other': other}
print({k: len(v) for k, v in sets.items()}, f'({time.time() - t:.0f}초)')

CAND_PATH = f'{OUT_DIR}/candidates.npz'
if os.path.exists(CAND_PATH):
    z = np.load(CAND_PATH)
    cand = {k: z[f'{k}_ids'] for k in sets}
    dense = {k: z[f'{k}_scores'] for k in sets}
    print('후보 불러옴')
else:
    emb_model = SentenceTransformer(EMB_MODEL, device='cuda').half()
    cand, dense = {}, {}
    for k, qs in sets.items():
        qe = emb_model.encode([QUERY_PREFIX + q['query'] for q in qs], batch_size=128,
                              normalize_embeddings=True, convert_to_numpy=True).astype('float32')
        D, I = index.search(qe, K)
        cand[k], dense[k] = I, D
    np.savez(CAND_PATH, **{f'{k}_ids': cand[k] for k in sets}, **{f'{k}_scores': dense[k] for k in sets})
    del emb_model; torch.cuda.empty_cache()

## 4. 재정렬 점수 계산 (모델별로 저장)

In [ ]:
def load_reranker(name, max_len):
    try:
        m = CrossEncoder(name, max_length=max_len, device='cuda', activation_fn=torch.nn.Identity())
    except TypeError:    # 예전 sentence-transformers
        m = CrossEncoder(name, max_length=max_len, device='cuda', default_activation_function=torch.nn.Identity())
    try:
        m.model.half()
    except AttributeError:
        m.half()
    return m

scores = {}
for label, (name, max_len, bs) in RERANKERS.items():
    path = f"{OUT_DIR}/scores_{name.split('/')[-1]}.npz"
    if os.path.exists(path):
        z = np.load(path); scores[label] = {k: z[k] for k in sets}
        print(f'{label}: 저장된 점수 불러옴'); continue
    t = time.time()
    model = load_reranker(name, max_len)
    scores[label] = {}
    for k, qs in sets.items():
        pairs = [(q['query'], texts[p]) for q, c in zip(qs, cand[k]) for p in c]
        s = model.predict(pairs, batch_size=bs, show_progress_bar=False, convert_to_numpy=True)
        scores[label][k] = np.asarray(s, dtype='float32').reshape(len(qs), K)
    np.savez(path, **scores[label])
    print(f'{label}: {time.time() - t:.0f}초')
    del model; torch.cuda.empty_cache()

## 5. 채점 함수 + 키워드 겹침 구간 (09와 같은 기준)

In [ ]:
def zs(m):   # 질문별 표준화 (모델끼리 점수 크기가 달라서)
    return (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-6)

def ranks_of(score_mat, k):
    out = []
    for q, ids, s in zip(sets[k], cand[k], score_mat):
        order = ids[np.argsort(-s)]
        gold = set(q['gold_pids'])
        out.append(next((r for r, p in enumerate(order, 1) if p in gold), None))
    return out

def metrics(ranks):
    ranks = [r if r is not None else 10**9 for r in ranks]
    return {'R@1': np.mean([r <= 1 for r in ranks]), 'R@5': np.mean([r <= 5 for r in ranks]),
            'R@10': np.mean([r <= 10 for r in ranks]), 'MRR@10': np.mean([1 / r if r <= 10 else 0 for r in ranks])}

STOP = set("""a an the of to in on for and or is are was were be been being do does did what which who whom whose
when where why how can could should would will shall may might must i you he she it we they me my your his her its our
their this that these those there here with without from by at as about into than then so if not no yes any some
much many more most less also just only very too long get gets got have has had having""".split())
def content_words(s):
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s.lower())
    return [w for w in s.split() if w not in STOP]
def overlap(q):
    qw = set(content_words(q['query']))
    return max(len(qw & set(content_words(texts[p]))) / len(qw) for p in q['gold_pids']) if qw else 0.0
bucket = ['하' if o <= 0.5 else ('상' if o >= 0.999 else '중') for o in map(overlap, test_set)]
print(Counter(bucket))

## 6. 단일 모델 + 합치기 설정 (검증으로 선택)

In [ ]:
def combo(cfg, k):
    # cfg = (모델 목록, α): 표준화 점수 평균 + α × 표준화 dense
    models, a = cfg
    m = np.mean([zs(scores[x][k]) for x in models], axis=0)
    return m + a * zs(dense[k]) if a else m

configs = {}
for x in RERANKERS:
    configs[x] = ((x,), 0.0)
val_mrr = lambda cfg: metrics(ranks_of(combo(cfg, 'val'), 'val'))['MRR@10']

# 재정렬 + dense: 모델마다 α 선택
for x in RERANKERS:
    a = max(ALPHAS, key=lambda a: val_mrr(((x,), a)))
    if a:
        configs[f'{x} + dense (α={a})'] = ((x,), a)

# 2개 앙상블: 모든 쌍 중 검증 최고 + 그 쌍에 dense
best_pair = max(itertools.combinations(RERANKERS, 2), key=lambda p: val_mrr((p, 0.0)))
configs[f'앙상블 {best_pair[0]} + {best_pair[1]}'] = (best_pair, 0.0)
a = max(ALPHAS, key=lambda a: val_mrr((best_pair, a)))
if a:
    configs[f'앙상블 {best_pair[0]} + {best_pair[1]} + dense (α={a})'] = (best_pair, a)

val_tab  = pd.DataFrame({n: metrics(ranks_of(combo(c, 'val'), 'val')) for n, c in configs.items()}).T.round(3)
test_ranks = {n: ranks_of(combo(c, 'test'), 'test') for n, c in configs.items()}
test_tab = pd.DataFrame({n: metrics(r) for n, r in test_ranks.items()}).T.round(3)
BEST = val_tab['MRR@10'].idxmax()
test_tab['검증 MRR@10'] = val_tab['MRR@10']
test_tab = test_tab.sort_values('검증 MRR@10', ascending=False)
print('검증 MRR@10 기준 최종 선택:', BEST)
display(test_tab)

## 7. 기준 vs 최종 선택: 구간별, 실패 유형, 통계적 차이

- **실패 유형**: 상위 10 밖인데 후보 100 안에 있음(재정렬 실패) vs 후보에도 없음 (후보는 같으니 '후보에도 없음'은 모든 모델이 같음)
- **95% 신뢰구간**: 질문을 다시 뽑아(부트스트랩 2,000번) MRR@10 차이 (최종 − 기준)

In [ ]:
buckets = ['하', '중', '상']
def by_bucket(ranks):
    return {b: metrics([r for r, bb in zip(ranks, bucket) if bb == b])['R@10'] for b in buckets}
show = [BASE, BEST] if BEST != BASE else [BASE]
bucket_tab = pd.DataFrame({n: by_bucket(test_ranks[n]) for n in show}).T.round(3)
print('구간별 R@10'); display(bucket_tab)

fail_rows = []
for n in show:
    for b in buckets:
        rs = [r for r, bb in zip(test_ranks[n], bucket) if bb == b]
        fail_rows.append({'설정': n, '구간': b, '상위 10 안': np.mean([r is not None and r <= 10 for r in rs]),
                          '후보 안, 순위 밖': np.mean([r is not None and r > 10 for r in rs]),
                          '후보에도 없음': np.mean([r is None for r in rs])})
fail_tab = pd.DataFrame(fail_rows).set_index(['설정', '구간']).round(3)
display(fail_tab)

rr = lambda ranks: np.array([1 / r if r is not None and r <= 10 else 0 for r in ranks])
diff = rr(test_ranks[BEST]) - rr(test_ranks[BASE])
rng = np.random.default_rng(42)
boots = [diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(2000)]
ci = (float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5)))
print(f'MRR@10 차이 ({BEST} − 기준): {diff.mean():+.3f}, 95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]')

## 8. 거절 구분력: 답 있음 vs 답 없음 / 분야 밖 (재정렬 최고 점수 AUC)

지금 챗봇은 재정렬 최고 점수가 기준보다 낮으면 거절합니다. 이 점수가 답 있음과 없음을 더 잘 가르면 거절도 좋아집니다. (단일 모델 원점수 기준, 08의 MiniLM-L-6은 0.80)

In [ ]:
from sklearn.metrics import roc_auc_score
auc_rows = {}
for x in RERANKERS:
    top = {k: scores[x][k].max(1) for k in ('test', 'noans', 'other')}
    auc_rows[x] = {
        '답 있음 vs 답 없음': roc_auc_score([1] * len(top['test']) + [0] * len(top['noans']),
                                         np.r_[top['test'], top['noans']]),
        '답 있음 vs 분야 밖': roc_auc_score([1] * len(top['test']) + [0] * len(top['other']),
                                         np.r_[top['test'], top['other']]),
    }
auc_tab = pd.DataFrame(auc_rows).T.round(3)
display(auc_tab)

## 9. 저장

In [ ]:
stamp = time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))
summary = '\n'.join([
    f'# 11 재정렬 개선 (v2.1) ({stamp} KST)', '',
    f'- 같은 후보: bge dense 상위 {K}개 / 시험 {len(test_set)} (답 있는 건강) / 검증 {len(val_set)} / 답 없음 {len(noans)} / 분야 밖 {len(other)}',
    '- 합치기: 질문별 표준화 점수. α는 dense 가중치, 앙상블 쌍과 α는 검증 MRR@10으로 선택',
    f'- 검증 MRR@10 기준 최종 선택: **{BEST}**',
    f'- MRR@10 차이 (최종 − 기준): {diff.mean():+.3f}, 95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]', '',
    '## 시험 500 (검증 MRR@10 순)', '', test_tab.to_markdown(), '',
    '## 겹침 구간별 R@10', '', bucket_tab.to_markdown(), '',
    '## 실패 유형', '', fail_tab.to_markdown(), '',
    '## 거절 구분력 (재정렬 최고 점수 AUC)', '', auc_tab.to_markdown(),
])
for path in (f'{OUT_DIR}/results_summary.md', f'{REPORT_DIR}/results_summary.md'):
    with open(path, 'w', encoding='utf-8') as f:
        f.write(summary)
with open(f'{OUT_DIR}/best_config.json', 'w', encoding='utf-8') as f:
    json.dump({'best': BEST, 'models': [RERANKERS[x][0] for x in configs[BEST][0]], 'alpha_dense': configs[BEST][1]},
              f, ensure_ascii=False, indent=2)
print(summary)

## 10. 드라이브 저장 마무리 + 런타임 종료

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()